# What Do Gardeners Ask? Text-Mining 14 Years of Gardening Q&A

**Questions:** What do people ask an online gardening community, when do they ask it, and how well has the community kept up with answering?

**Data:** The public [Stack Exchange Data Dump](https://archive.org/details/stackexchange) for *Gardening & Landscaping Stack Exchange* (July 2010 – March 2024), licensed CC BY-SA 4.0. Two tables are used:

| Table | Contents | Rows |
|---|---|---|
| `Posts` | Every question and answer: title, HTML body, tags, score, dates | 44,155 |
| `Users` | User profiles, including a free-text location field | 29,094 |

**Techniques:** XML → CSV conversion, regular expressions for text extraction (tags, photos, locations, keyword search), time-series aggregation, and interactive Bokeh charts.

## 1. Setup

In [1]:
import re
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import pycountry

from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.layouts import column
from bokeh.models import (ColumnDataSource, HoverTool, LinearAxis, Range1d, TextInput, CustomJS,
                          Div, Legend, NumeralTickFormatter, LinearColorMapper, ColorBar)
from bokeh.transform import transform
from bokeh.palettes import OrRd9

output_notebook(hide_banner=True)
pd.set_option("display.float_format", "{:,.2f}".format)
RAW = Path("data/raw")
GREEN, ORANGE, GREY = "#2f855a", "#dd6b20", "#a0aec0"

## 2. Load the data (XML → CSV)

Stack Exchange publishes its dumps as XML, with one `<row>` element per record and the fields stored as attributes. The converter below turns each file into a CSV once, and later runs read the CSV directly.

In [2]:
def xml_to_csv(xml_path, csv_path):
    """Convert a Stack Exchange XML dump file (<row ... /> elements) to CSV."""
    rows = [el.attrib for el in ET.parse(xml_path).getroot().iter("row")]
    pd.DataFrame(rows).to_csv(csv_path, index=False)

def load(name):
    csv, xml = RAW / f"{name}.csv", RAW / f"{name}.xml"
    if not csv.exists():
        if not xml.exists():
            raise FileNotFoundError(f"Put {name}.xml from the gardening.stackexchange.com dump in {RAW}/ (see README)")
        xml_to_csv(xml, csv)
    return pd.read_csv(csv, low_memory=False)

posts = load("Posts")
users = load("Users")
print(f"Posts: {posts.shape}   Users: {users.shape}")

Posts: (44155, 22)   Users: (29094, 12)


## 3. Clean and structure

`Posts` mixes several record types in one table, identified by `PostTypeId`. Questions (1) and answers (2) have different columns: answers have no title or tags, they point to their question through `ParentId`. So rather than dropping rows with missing titles (which would silently delete every answer), I split the table by type first.

In [3]:
posts["CreationDate"] = pd.to_datetime(posts["CreationDate"])
print(posts["PostTypeId"].value_counts().rename({1: "Question", 2: "Answer", 4: "Tag wiki excerpt",
                                                  5: "Tag wiki", 6: "Moderator nomination", 7: "Wiki placeholder"}))

q = posts[posts["PostTypeId"] == 1].copy()
a = posts[posts["PostTypeId"] == 2].copy()
q = q[["Id", "CreationDate", "Title", "Body", "Tags", "Score", "ViewCount", "AnswerCount", "AcceptedAnswerId"]]
a = a[["Id", "ParentId", "CreationDate", "OwnerUserId", "Score"]]
print(f"\n{len(q):,} questions, {len(a):,} answers")
print("Questions with missing title/body/tags:", q[["Title", "Body", "Tags"]].isna().any(axis=1).sum())

PostTypeId
Answer                  25421
Question                17860
Tag wiki                  433
Tag wiki excerpt          433
Moderator nomination        6
Wiki placeholder            2
Name: count, dtype: int64

17,860 questions, 25,421 answers
Questions with missing title/body/tags: 0


Every question is complete, so no rows need dropping. Next I derive the fields the analysis needs:

In [4]:
# When was each question first answered?
first_answer = a.groupby("ParentId")["CreationDate"].min()
q["first_answer"] = q["Id"].map(first_answer)
q["hours_to_answer"] = (q["first_answer"] - q["CreationDate"]).dt.total_seconds() / 3600
q["answered"] = q["AnswerCount"] > 0
q["accepted"] = q["AcceptedAnswerId"].notna()
q["year"] = q["CreationDate"].dt.year
q["month"] = q["CreationDate"].dt.month

# Regex 1: tags are stored as one string like "|tomatoes|pests|", so extract them into a list
q["tag_list"] = q["Tags"].str.findall(r"\|([^|]+)(?=\|)")

# Regex 2: does the question body contain an embedded photo?
q["has_photo"] = q["Body"].str.contains(r"<img\b", case=False, regex=True)

q[["Title", "tag_list", "has_photo", "answered", "hours_to_answer"]].head()

,Title,tag_list,has_photo,answered,hours_to_answer
0,Which side of a house will get the most sunlight?,"[houseplants, sunlight]",False,True,0.05
1,Why do my upside-down tomato plants curve up a...,[tomatoes],False,True,0.07
2,How can I revive a tree that's been stripped o...,"[trees, tree-care]",False,True,0.25
3,What's the most effective way to harvest rhubarb?,"[rhubarb, vegetables, harvesting]",False,True,0.04
5,What is a fast-growing deciduous tree for shad...,"[trees, shade]",False,True,0.26


---
## 4. Is the community keeping up?
### Chart 1: Question volume and answer rate by year

In [5]:
yearly = (q[q["year"].between(2011, 2023)]          # 2010 has 3 questions; 2024 is a partial year
          .groupby("year")
          .agg(questions=("Id", "size"), answered=("answered", "mean"),
               accepted=("accepted", "mean"), median_hours=("hours_to_answer", "median")))
yearly

,questions,answered,accepted,median_hours
year,,,,
2011,886,0.99,0.77,2.61
2012,836,0.99,0.65,8.28
2013,693,0.98,0.54,7.94
2014,1057,0.98,0.64,4.57
2015,1282,0.96,0.55,4.57
2016,2007,0.95,0.52,3.54
2017,1888,0.91,0.42,3.59
2018,2032,0.85,0.37,3.52
2019,1950,0.79,0.33,4.22


In [6]:
src1 = ColumnDataSource(yearly.reset_index().assign(year_s=lambda d: d["year"].astype(str),
                                                     ans_pct=lambda d: d["answered"] * 100,
                                                     acc_pct=lambda d: d["accepted"] * 100))
years = [str(y) for y in yearly.index]

fig1 = figure(x_range=years, height=420, width=800, title="Questions peaked in 2016–2020, but the share answered fell steadily",
              y_axis_label="Questions asked", toolbar_location=None, tools="")
bars = fig1.vbar(x="year_s", top="questions", source=src1, width=0.7, color="#c6f6d5", line_color=None)
fig1.y_range.start = 0
fig1.extra_y_ranges = {"pct": Range1d(0, 105)}
fig1.add_layout(LinearAxis(y_range_name="pct", axis_label="% of questions"), "right")
l1 = fig1.line("year_s", "ans_pct", source=src1, y_range_name="pct", line_width=3, color=GREEN)
fig1.scatter("year_s", "ans_pct", source=src1, y_range_name="pct", size=7, color=GREEN)
l2 = fig1.line("year_s", "acc_pct", source=src1, y_range_name="pct", line_width=3, color=ORANGE, line_dash="dashed")
fig1.add_layout(Legend(items=[("Questions asked", [bars]), ("% answered", [l1]), ("% with accepted answer", [l2])],
                       location="top_right"), "below")
fig1.legend.orientation = "horizontal"
fig1.add_tools(HoverTool(renderers=[bars], tooltips=[("Year", "@year_s"), ("Questions", "@questions{0,0}"),
               ("Answered", "@ans_pct{0}%"), ("Accepted answer", "@acc_pct{0}%"), ("Median wait", "@median_hours{0.0} h")]))
fig1.xgrid.grid_line_color = None
show(fig1)

- **Answer rate fell from 99% (2011–12) to 63% (2023).** More than a third of recent questions never get a single answer.
- Accepted answers fell even faster (77% → 19%), and the median wait for a first answer rose from ~3 hours to ~8–10 hours.
- Volume peaked at ~2,000 questions a year in 2016–2020, then halved by 2022. This is consistent with the widely reported drop in activity across Stack Exchange sites since 2020.
- *Caveat:* recent questions have had less time to collect answers, but the decline is steady from 2015 onward, so it isn't just a recency effect.

### Chart 2: Who does the answering?

In [7]:
per_user = a["OwnerUserId"].dropna().value_counts().sort_values(ascending=False).to_numpy()
cum_answers = np.concatenate([[0], per_user.cumsum() / per_user.sum() * 100])
cum_users = np.linspace(0, 100, len(cum_answers))

top1 = per_user[: max(1, len(per_user) // 100)].sum() / per_user.sum() * 100
top10_users = per_user[:10].sum() / per_user.sum() * 100
one_timers = (per_user == 1).mean() * 100
print(f"{len(per_user):,} people have answered. Top 1% wrote {top1:.0f}% of answers; the top 10 people wrote {top10_users:.0f}%.")
print(f"{one_timers:.0f}% of answerers only ever wrote one answer.")

3,743 people have answered. Top 1% wrote 61% of answers; the top 10 people wrote 42%.
76% of answerers only ever wrote one answer.


In [8]:
src2 = ColumnDataSource(dict(users=cum_users, answers=cum_answers))
fig2 = figure(height=420, width=800, x_range=(0, 100), y_range=(0, 101),
              title=f"The top 1% of answerers wrote {top1:.0f}% of all answers",
              x_axis_label="% of answerers (most active first)", y_axis_label="% of all answers written",
              tools="", toolbar_location=None)
fig2.varea(x="users", y1=0, y2="answers", source=src2, fill_color=GREEN, fill_alpha=0.15)
fig2.line("users", "answers", source=src2, line_width=3, color=GREEN)
fig2.line([0, 100], [0, 100], line_dash="dashed", color=GREY, legend_label="If everyone answered equally")
fig2.scatter([1], [top1], size=10, color=ORANGE)
fig2.text([3], [top1 - 6], text=[f"Top 1% → {top1:.0f}% of answers"], text_font_size="10pt", text_color="#2d3748")
fig2.add_tools(HoverTool(tooltips=[("Top", "@users{0.0}% of answerers"), ("Wrote", "@answers{0.0}% of answers")], mode="vline"))
fig2.legend.location = "bottom_right"
show(fig2)

- The community runs on a handful of experts: **the top 1% of answerers wrote ~61% of all answers, and just 10 people wrote 42%.**
- Three in four answerers contributed only once.
- This concentration explains Chart 1. When a few core contributors become less active, the answer rate drops quickly because there's no broad base to replace them.
- *(Individuals are deliberately not named. The point is the structure, not the people.)*

---
## 5. What do people ask about?
### Chart 3: Top topics, and which ones go unanswered

In [9]:
tags = (q.explode("tag_list")
          .groupby("tag_list")
          .agg(questions=("Id", "size"), unanswered=("answered", lambda s: (1 - s.mean()) * 100))
          .sort_values("questions", ascending=False))
top_tags = tags.head(15).iloc[::-1].reset_index()
overall_unanswered = (1 - q["answered"].mean()) * 100
tags.head(15)

,questions,unanswered
tag_list,,
identification,3237,11.43
trees,1654,14.33
houseplants,1535,17.85
diagnosis,1463,21.80
plant-health,1232,20.70
plant-care,889,18.11
diseases,838,28.28
leaves,826,25.30
vegetables,799,9.39


In [10]:
src3 = ColumnDataSource(top_tags)
mapper = LinearColorMapper(palette=OrRd9[::-1][2:], low=5, high=30)
fig3 = figure(y_range=list(top_tags["tag_list"]), height=480, width=800,
              title="Plant ID is the #1 question type; disease questions are the most likely to go unanswered",
              x_axis_label="Number of questions", tools="", toolbar_location=None)
fig3.hbar(y="tag_list", right="questions", source=src3, height=0.7,
          fill_color=transform("unanswered", mapper), line_color=None)
fig3.add_layout(ColorBar(color_mapper=mapper, title="% unanswered", width=10), "right")
fig3.add_tools(HoverTool(tooltips=[("Tag", "@tag_list"), ("Questions", "@questions{0,0}"), ("Unanswered", "@unanswered{0.0}%")]))
fig3.x_range.start = 0
fig3.ygrid.grid_line_color = None
show(fig3)

- **"What is this plant?" dominates:** `identification` is the top tag, at 3,237 questions (18% of all questions).
- **Plant problems are the hardest to get answered:** `diseases` (28% unanswered), `leaves` (25%) and `diagnosis` (22%) sit well above the overall rate (~14%). Practical tags like `flowers`, `lawn` and `vegetables` go unanswered less than 10% of the time.
- So the community is good at naming plants and giving how-to advice, but struggles with remote diagnosis, where a photo and description often aren't enough.

### Chart 4: When do people ask?

In [11]:
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
monthly = q.groupby("month").size().reindex(range(1, 13))
src4 = ColumnDataSource(dict(month=month_names, questions=monthly.to_numpy()))
fig4 = figure(x_range=month_names, height=380, width=800, tools="", toolbar_location=None,
              title=f"Questions peak in May–June: {monthly.max() / monthly.min():.1f}× more than in December",
              y_axis_label="Questions (2010–2024, all years)")
fig4.vbar(x="month", top="questions", source=src4, width=0.75, color=GREEN)
fig4.add_tools(HoverTool(tooltips=[("Month", "@month"), ("Questions", "@questions{0,0}")]))
fig4.y_range.start = 0
fig4.xgrid.grid_line_color = None
show(fig4)

- Activity follows the **Northern Hemisphere growing season**, peaking in May–June (spring planting and early pests) and bottoming out in December.
- That fits Chart 6 below: most users with a stated location are in the US, UK, Canada and Europe.

### Chart 5: Seasonal keyword explorer (regex search)
Type any regular expression to see what share of question titles match it in each calendar month. It runs in the browser with no Python needed.

In [12]:
titles = q["Title"].tolist()
months = (q["month"] - 1).tolist()
totals = monthly.tolist()

def share_by_month(pattern):
    hits = q["Title"].str.contains(pattern, case=False, regex=True)
    return (q[hits].groupby("month").size().reindex(range(1, 13), fill_value=0) / monthly * 100).round(2).tolist()

for pat in ["tomato", "frost|winter|cold", "seed|germinat"]:
    s = share_by_month(pat)
    print(f"{pat:<20} peak {month_names[int(np.argmax(s))]} ({max(s):.1f}%), low {month_names[int(np.argmin(s))]} ({min(s):.1f}%)")

tomato               peak May (4.0%), low Mar (1.6%)
frost|winter|cold    peak Nov (5.8%), low Jul (0.4%)
seed|germinat        peak Apr (7.5%), low Jan (4.1%)


In [13]:
default = "frost|winter|cold"
src5 = ColumnDataSource(dict(month=month_names, share=share_by_month(default)))
fig5 = figure(x_range=month_names, height=380, width=800, tools="", toolbar_location=None,
              title=f"Share of question titles matching /{default}/", y_axis_label="% of that month's questions")
fig5.line("month", "share", source=src5, line_width=3, color=ORANGE)
fig5.scatter("month", "share", source=src5, size=8, color=ORANGE)
fig5.add_tools(HoverTool(tooltips=[("Month", "@month"), ("Share", "@share{0.0}%")]))
fig5.y_range.start = 0
fig5.xgrid.grid_line_color = None

box = TextInput(value=default, title="Regex (case-insensitive), e.g. tomato | seed|germinat | aphid|pest", width=420)
box.js_on_change("value", CustomJS(args=dict(src=src5, box=box, fig=fig5, titles=titles, months=months, totals=totals), code="""
    let re;
    try { re = new RegExp(box.value, "i"); } catch (e) { fig.title.text = "Invalid regex"; return; }
    const hits = new Array(12).fill(0);
    for (let i = 0; i < titles.length; i++) if (re.test(titles[i])) hits[months[i]] += 1;
    src.data = {month: src.data.month, share: hits.map((h, i) => Math.round(h / totals[i] * 1000) / 10)};
    fig.title.text = `Share of question titles matching /${box.value}/`;
"""))
show(column(box, fig5))

- `frost|winter|cold` is about **14× more common in November than in July**, and `tomato` peaks in May–July. The text clearly tracks the gardening calendar.
- Seed and germination questions peak twice: in spring (Mar–Apr) and again in November.
- `identif` stays fairly flat all year (~8–14%). Plant ID is a year-round need.

### Chart 6: Where are users from? (regex location parsing)
`Location` is free text ("Buffalo, NY", "Coimbatore, Tamil Nadu, India", "On a pool table."). I take the last comma-separated part with a regex, then map it to a country: US state codes and names → United States, common aliases (UK, England, USA…), then an ISO lookup with `pycountry`.

In [14]:
US_STATE_CODES = {"AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA","KS","KY","LA","ME","MD",
                  "MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC",
                  "SD","TN","TX","UT","VT","VA","WA","WV","WI","WY","DC"}
US_STATE_NAMES = {s.name.lower() for s in pycountry.subdivisions.get(country_code="US")}
CA_PROVINCES = {"ON","BC","AB","QC","MB","SK","NS","NB","NL","PE"}
ALIASES = {"usa": "United States", "us": "United States", "united states of america": "United States",
           "uk": "United Kingdom", "england": "United Kingdom", "scotland": "United Kingdom", "wales": "United Kingdom",
           "northern ireland": "United Kingdom", "great britain": "United Kingdom", "the netherlands": "Netherlands",
           "holland": "Netherlands", "russia": "Russian Federation"}

def to_country(location):
    parts = [p.strip() for p in re.split(r"[,/|–-]", location) if p.strip()]
    if not parts:
        return None
    last = re.sub(r"[^A-Za-z .]", "", parts[-1]).strip().rstrip(".")     # keep letters only
    if not last:
        return None
    if len(last) == 2 and last.upper() in US_STATE_CODES:
        return "United States"
    if len(last) == 2 and last.upper() in CA_PROVINCES:
        return "Canada"
    if last.lower() in US_STATE_NAMES:
        return "United States"
    if last.lower() in ALIASES:
        return ALIASES[last.lower()]
    try:
        return pycountry.countries.lookup(last).name
    except LookupError:
        return None

locations = users["Location"].dropna().astype(str)
unique_map = {loc: to_country(loc) for loc in locations.unique()}
countries = locations.map(unique_map)
print(f"{users['Location'].notna().mean():.0%} of users filled in a location; {countries.notna().mean():.0%} of those mapped to a country")
country_share = (countries.value_counts(normalize=True) * 100).head(12)
country_share

32% of users filled in a location; 83% of those mapped to a country


Location
United States    49.86
United Kingdom   10.31
India             5.64
Canada            4.96
Australia         3.63
Germany           3.02
Netherlands       1.31
France            1.28
Indonesia         0.90
New Zealand       0.89
South Africa      0.80
Italy             0.71
Name: proportion, dtype: float64

In [15]:
cs = country_share.iloc[::-1].reset_index()
cs.columns = ["country", "share"]
src6 = ColumnDataSource(cs)
fig6 = figure(y_range=list(cs["country"]), height=420, width=800, tools="", toolbar_location=None,
              title="Half of located users are in the US; 4 of the top 5 countries are English-speaking",
              x_axis_label="% of users with a recognisable location")
fig6.hbar(y="country", right="share", source=src6, height=0.7, color=GREEN)
fig6.add_tools(HoverTool(tooltips=[("Country", "@country"), ("Share", "@share{0.0}%")]))
fig6.x_range.start = 0
fig6.ygrid.grid_line_color = None
show(fig6)

- The **US (~50%), UK (~10%), India, Canada and Australia** lead. The platform's advice is weighted toward temperate Northern Hemisphere conditions.
- Only about a third of users give a location at all, so this describes the people who chose to share one, not the whole user base.

### Bonus check: do photos help a question get answered?

In [16]:
overall = q.groupby("has_photo")["answered"].mean() * 100
within = (q[q["year"].between(2014, 2023)].groupby(["year", "has_photo"])["answered"].mean().unstack() * 100)
print("Overall answer rate: with photo {:.1f}%, without {:.1f}%".format(overall[True], overall[False]))
print("Average within-year difference (photo − no photo), 2014–2023: {:+.1f} points".format((within[True] - within[False]).mean()))
print("Share of questions with a photo: {:.0%} (2012) → {:.0%} (2023)".format(
      q[q.year == 2012]["has_photo"].mean(), q[q.year == 2023]["has_photo"].mean()))

Overall answer rate: with photo 84.5%, without 86.9%
Average within-year difference (photo − no photo), 2014–2023: +1.5 points
Share of questions with a photo: 28% (2012) → 62% (2023)


On the raw numbers, photo questions look *less* likely to be answered (84% vs 87%). But photos became far more common over time (28% of questions in 2012 vs 62% in 2023), while answer rates were falling. Compared **within the same year**, photo questions do slightly *better* (about +1.5 points). The raw comparison was confounded by time, a small example of why it pays to check for confounders before reporting a headline number.

---
## 6. Summary

| Finding | Evidence |
|---|---|
| Answer rate fell from 99% (2011) to 63% (2023); accepted answers 77% → 19% | Chart 1 |
| The top 1% of answerers wrote ~61% of all answers; 10 people wrote 42% | Chart 2 |
| Plant ID is the #1 topic (18% of questions); disease and diagnosis questions are the most often unanswered (22–28%) | Chart 3 |
| Questions peak in May–June, 2.6× December; "frost/winter" is 14× more common in November than July | Charts 4–5 |
| ~Half of located users are in the US; 4 of the top 5 countries are English-speaking | Chart 6 |
| Photos help slightly (+1.5 pts) once you control for year; the raw comparison points the wrong way | Bonus check |

## 7. Ethics & limitations
- **Licence:** Stack Exchange content is CC BY-SA 4.0. The source is credited and no raw content is redistributed in this repo.
- **Privacy:** no usernames, user IDs or individual locations are shown, only aggregates. Some users typed full street addresses into their location field. That's exactly why raw locations are reduced to country level and never displayed.
- **Selection bias:** the data covers people who post in English on one Q&A site. Location data covers only the ~32% who shared one.
- **Regex limits:** the location parser misses joke entries and city-only locations ("Seattle", "NYC"), and title keyword matching can't tell context ("cold frame" ≠ cold weather). Patterns were spot-checked, but the counts are approximate.